## Teste de hipótese

### Dataset Digits

In [1]:
import numpy as np
from scipy import stats

from sklearn import datasets
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [2]:
d = datasets.load_digits()
print('Conjunto "Digits"',
      '\nNúmero de amostras:', d.data.shape[0],
      '\nClasses presentes:', *d.target_names)

Conjunto "Digits" 
Número de amostras: 1797 
Classes presentes: 0 1 2 3 4 5 6 7 8 9


In [3]:
# estimadores a serem comparados
r = np.random.RandomState(42)

experiments = 5
estimators = [
    SVC(kernel='linear'),
    LogisticRegression()
]

In [4]:
# execução dos experimentos
scores = []

for experiment in range(experiments):
    print('Executando experimento #%i' % experiment)
    experiment_seed = r.randint(100)

    # Compartilha o StratifiedKFold entre os estimadores,
    # garantindo que ambos sejam executados sobre as mesmas folds.
    skf = StratifiedKFold(n_splits=2,
                          shuffle=True,
                          random_state=experiment_seed)
    exp_scores = []
    
    for e in estimators:
        print('- Testando', e)
        score = cross_val_score(e, d.data, d.target, cv=skf, n_jobs=4)
        exp_scores += [score]
        print('  pontuação nos folds:', score, end='\n\n')

    scores += [exp_scores]
    print()

scores = np.asarray(scores)

Executando experimento #0
- Testando SVC(C=1.0, cache_size=200, class_weight=None, coef0=0.0,
  decision_function_shape='ovr', degree=3, gamma='auto', kernel='linear',
  max_iter=-1, probability=False, random_state=None, shrinking=True,
  tol=0.001, verbose=False)
  pontuação nos folds: [0.96781354 0.97991071]

- Testando LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='ovr', n_jobs=1,
          penalty='l2', random_state=None, solver='liblinear', tol=0.0001,
          verbose=0, warm_start=False)
  pontuação nos folds: [0.95116537 0.95200893]


Executando experimento #1
- Testando SVC(C=1.0, cache_size=200, class_weight=None, coef0=0.0,
  decision_function_shape='ovr', degree=3, gamma='auto', kernel='linear',
  max_iter=-1, probability=False, random_state=None, shrinking=True,
  tol=0.001, verbose=False)
  pontuação nos folds: [0.97669256 0.9765625 ]

- Testando LogisticRegression(C=1.0, class_weight

In [5]:
# Constrói um vetor com os nomes das classes dos estimadores em `estimators`.
names = [e.__class__.__name__ for e in estimators]

In [6]:
# Seleciona as pontuações de todos os experimentos, estimador SVC, todos os folds.
scores_a = scores[:, 0, :].flatten()
# Seleciona as pontuações de todos os experimentos, estimador LR, todos os folds.
scores_b = scores[:, 1, :].flatten()

In [7]:
if scores_a.mean() > scores_b.mean():
    best = names[0]
else:
    best = names[1]

print('Em média, o estimador', best, 'obteve uma melhor pontuação:')
for name, s in zip(names, (scores_a, scores_b)):
    print(name, '=', s.mean())

Em média, o estimador SVC obteve uma melhor pontuação:
SVC = 0.9756274030838752
LogisticRegression = 0.953143456675123


In [8]:
s, p = stats.wilcoxon(scores_a, scores_b)

# `p` representa a probabilidade de que a diferença de 
# pontuação observada tenha ocorrido ao acaso.
if p < 0.05:
    print('A diferença em pontuação é significativa (p-valor: %s < 5%%)!' % p)
else:
    print('Entretanto, a diferença em pontuação não é '
          'significativa (p-valor: %s >= 5%%).' % p)

A diferença em pontuação é significativa (p-valor: 0.005005074402905223 < 5%)!
